In [1]:
!pip install xplique

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/ConceptualizingConceptDrift/')

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

import torch
import numpy as np

from sklearn.decomposition import NMF, non_negative_factorization
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

import random

from experiment_helpers.helper_function import *
from experiment_helpers.driftLocalizer import Localizer
from text_helpers.CraftText import CraftText, CraftTextCombined, full_text_activations
from text_helpers.CustomBertModel import CustomBertForSequenceClassification

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

device = 'cuda'

checkpoint_name = "fabriceyhc/bert-base-uncased-dbpedia_14"
model = CustomBertForSequenceClassification.from_pretrained(checkpoint_name, num_labels=14)
model = model.eval().to(device)
tokenizer = AutoTokenizer.from_pretrained(checkpoint_name)

dbpedia = load_dataset("fancyzhx/dbpedia_14")

all_texts = list(dbpedia["train"]["content"])
all_labels = np.array(dbpedia["train"]["label"])

class_ids = [2, 3, 4, 11, 12, 13]

'''
2 = artist
3 = athlete
4 = office holder
11 = album
12 = film
13 = written work
'''

subset_mask = np.isin(all_labels, class_ids)
texts = [t for t, m in zip(all_texts, subset_mask) if m]
labels = all_labels[subset_mask]

keys = class_ids

print(len(texts))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

240000


In [5]:
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [6]:
import random

np.random.seed(42)
random.seed(42)

patch_mode = "window"
win_size = 15
stride = 10


label_maps = []
drift_ratios = []
drift_localizer = []

one_local_one_global_l = []
one_local_l = []
two_local_l = []
three_local_l = []
one_global_l = []
two_global_l = []
three_global_l = []

one_local_one_global_preds_l = []
one_local_preds_l = []
two_local_preds_l = []
three_local_preds_l = []
one_global_preds_l = []
two_global_preds_l = []
three_global_preds_l = []

one_local_l_probs = []
two_local_l_probs = []
three_local_l_probs = []
one_local_preds_l_probs = []
two_local_preds_l_probs = []
three_local_preds_l_probs = []

reconstructed_single_concepts = []
reconstructed_single_concepts_preds = []
reconstructed_2_concepts = []
reconstructed_2_concepts_preds = []
reconstructed_3_concepts = []
reconstructed_3_concepts_preds = []
reconstructed_all_concepts = []
reconstructed_all_concepts_preds = []

run_num = 25

for j in range(run_num):

    sample_ids = np.random.choice(len(texts), 500, False)

    sample_texts = [texts[i] for i in sample_ids]

    keys_shuffled = keys.copy()
    random.shuffle(keys_shuffled)

    initial_labels = [0, 1, 2]
    random.shuffle(initial_labels)

    label_map = {keys_shuffled[i]: initial_labels[i] for i in range(3)}

    for i in range(3, len(keys_shuffled)):
        label_map[keys_shuffled[i]] = random.randint(0, 2)

    label_maps.append(label_map)

    labels_mapped = np.array([label_map[class_id] for class_id in labels])

    drift_labels = labels_mapped[sample_ids]

    label_2_idx = np.where(drift_labels == 2)[0]
    y_mixed = drift_labels.copy()
    y_mixed[label_2_idx] = np.random.choice([0, 1], size=len(label_2_idx))

    sample_labels = y_mixed

    drift_ratios.append({"BD": len(np.where(drift_labels == 0)[0]),
                         "AD": len(np.where(drift_labels == 1)[0]),
                         "Both": len(np.where(drift_labels == 2)[0])})

    patch_act = full_text_activations(sample_texts, model, tokenizer, device=device)
    train_labels = sample_labels

    bd_indices = np.where(sample_labels != 1)[0]
    ad_indices = np.where(sample_labels != 0)[0]

    bd_texts = [sample_texts[i] for i in bd_indices]
    bd_labels = [sample_labels[i] for i in bd_indices]
    ad_texts = [sample_texts[i] for i in ad_indices]
    ad_labels = [sample_labels[i] for i in ad_indices]

    bd_fit = CraftText(model_wrapper=model, tokenizer=tokenizer, num_concepts=10,
                       patch_mode=patch_mode, win_size=win_size, stride=stride, device=device)
    print("Fitting Unsupervised Craft....")
    bd_crops, bd_crops_u, bd_w = bd_fit.fit(bd_texts, bd_labels)

    ad_fit = CraftText(model_wrapper=model, tokenizer=tokenizer, num_concepts=10,
                       patch_mode=patch_mode, win_size=win_size, stride=stride, device=device)
    print("Fitting Unsupervised Craft....")
    ad_crops, ad_crops_u, ad_w = ad_fit.fit(ad_texts, ad_labels)

    drift_basis = np.vstack([bd_w, ad_w])

    drift_craft = CraftTextCombined(model_wrapper=model, tokenizer=tokenizer, basis=drift_basis,
                                    patch_mode=patch_mode, win_size=win_size, stride=stride, device=device)
    print("Fitting Craft....")
    drift_craft.transform_all(sample_texts, sample_labels)

    X_clean = patch_act
    y_clean = train_labels

    localizer_model = Localizer()

    X_train_clean, X_test_clean, y_train, y_test = \
        train_test_split(X_clean, y_clean, train_size=0.7, random_state=42)

    print('Fitting Random Forest classifier...')
    localizer_model.fit(X_train_clean, y_train)
    print('Fitting complete.')

    localizer_bin_preds = localizer_model.l_predict(X_test_clean)
    drift_localizer.append(accuracy_score(localizer_bin_preds, y_test))

    drift_imp = np.round(estimate_importance_l(localizer_model, drift_craft, drift_basis, X_train_clean), 3)

    image_drift_imp_l = [estimate_importance_helper_l(drift_craft, localizer_model, drift_basis,
                                                  image, class_of_interest=localizer_bin_preds[i])
                               for i, image in enumerate(X_test_clean)]

    one_local_one_global_l.append(local_one_imp_concept_globally_l(drift_craft, image_drift_imp_l, y_test))
    one_local_l.append(local_imp_concepts_globally_l(drift_craft, image_drift_imp_l, num=1, labels=y_test))
    two_local_l.append(local_imp_concepts_globally_l(drift_craft, image_drift_imp_l, num=2, labels=y_test))
    three_local_l.append(local_imp_concepts_globally_l(drift_craft, image_drift_imp_l, num=3, labels=y_test))

    one_global_l.append(global_imp_concepts_locally_l(drift_craft, image_drift_imp_l, num=1, labels=y_test))
    two_global_l.append(global_imp_concepts_locally_l(drift_craft, image_drift_imp_l, num=2, labels=y_test))
    three_global_l.append(global_imp_concepts_locally_l(drift_craft, image_drift_imp_l, num=3, labels=y_test))

    one_local_one_global_preds_l.append(local_one_imp_concept_globally_l(drift_craft, image_drift_imp_l, localizer_bin_preds))
    one_local_preds_l.append(local_imp_concepts_globally_l(drift_craft, image_drift_imp_l, num=1, labels=localizer_bin_preds))
    two_local_preds_l.append(local_imp_concepts_globally_l(drift_craft, image_drift_imp_l, num=2, labels=localizer_bin_preds))
    three_local_preds_l.append(local_imp_concepts_globally_l(drift_craft, image_drift_imp_l, num=3, labels=localizer_bin_preds))

    one_global_preds_l.append(global_imp_concepts_locally_l(drift_craft, image_drift_imp_l, num=1, labels=localizer_bin_preds))
    two_global_preds_l.append(global_imp_concepts_locally_l(drift_craft, image_drift_imp_l, num=2, labels=localizer_bin_preds))
    three_global_preds_l.append(global_imp_concepts_locally_l(drift_craft, image_drift_imp_l, num=3, labels=localizer_bin_preds))

    localizer_bin_train_preds = localizer_model.l_predict(X_train_clean)
    image_drift_imp_l_train = [estimate_importance_helper_l(drift_craft, localizer_model, drift_basis,
                                                  image, class_of_interest=localizer_bin_train_preds[i])
                               for i, image in enumerate(X_train_clean)]
    concept_dist = concept_counter(image_drift_imp_l_train, localizer_bin_train_preds)

    one_local_l_probs.append(local_imp_concepts_probability(concept_dist, image_drift_imp_l, num=1, labels=y_test))
    two_local_l_probs.append(local_imp_concepts_probability(concept_dist, image_drift_imp_l, num=2, labels=y_test))
    three_local_l_probs.append(local_imp_concepts_probability(concept_dist, image_drift_imp_l, num=3, labels=y_test))

    one_local_preds_l_probs.append(local_imp_concepts_probability(concept_dist, image_drift_imp_l, num=1, labels=localizer_bin_preds))
    two_local_preds_l_probs.append(local_imp_concepts_probability(concept_dist, image_drift_imp_l, num=2, labels=localizer_bin_preds))
    three_local_preds_l_probs.append(local_imp_concepts_probability(concept_dist, image_drift_imp_l, num=3, labels=localizer_bin_preds))

    reconstructed_single_concept = reconstruct_inputs(X_test_clean, image_drift_imp_l, drift_basis, num_concepts=1)
    localizer_preds = localizer_model.l_predict(reconstructed_single_concept)
    reconstructed_single_concepts.append(accuracy_score(localizer_preds, y_test))
    reconstructed_single_concepts_preds.append(accuracy_score(localizer_preds, localizer_bin_preds))

    reconstructed_2_concept = reconstruct_inputs(X_test_clean, image_drift_imp_l, drift_basis, num_concepts=2)
    localizer_preds = localizer_model.l_predict(reconstructed_2_concept)
    reconstructed_2_concepts.append(accuracy_score(localizer_preds, y_test))
    reconstructed_2_concepts_preds.append(accuracy_score(localizer_preds, localizer_bin_preds))

    reconstructed_3_concept = reconstruct_inputs(X_test_clean, image_drift_imp_l, drift_basis, num_concepts=3)
    localizer_preds = localizer_model.l_predict(reconstructed_3_concept)
    reconstructed_3_concepts.append(accuracy_score(localizer_preds, y_test))
    reconstructed_3_concepts_preds.append(accuracy_score(localizer_preds, localizer_bin_preds))

    reconstructed_all_concept = reconstruct_inputs(X_test_clean, image_drift_imp_l, drift_basis, num_concepts=len(drift_basis))
    localizer_preds = localizer_model.l_predict(reconstructed_all_concept)
    reconstructed_all_concepts.append(accuracy_score(localizer_preds, y_test))
    reconstructed_all_concepts_preds.append(accuracy_score(localizer_preds, localizer_bin_preds))

    print("Run:", j)

Fitting Unsupervised Craft....


Fitting Unsupervised Craft....


Fitting Craft....


Fitting Random Forest classifier...
Determine optimal parameters using cross validation


low threshold: 0.15 Mean:0.3057142857142857 High threshold:0.5, No. Leaves:20


...

Fitting Craft....


Fitting Random Forest classifier...
Determine optimal parameters using cross validation


low threshold: 0.2 Mean:0.39714285714285713 High threshold:0.6, No. Leaves:20


Fitting complete.


Run: 24


In [ ]:
import csv


methods = [drift_localizer,
            one_local_one_global_l,
            one_local_l,
            two_local_l,
            three_local_l,
            one_local_l_probs,
            two_local_l_probs,
            three_local_l_probs,

            one_global_l,
            two_global_l,
            three_global_l,

            one_local_one_global_preds_l,
            one_local_preds_l,
            two_local_preds_l,
            three_local_preds_l,
            one_local_preds_l_probs,
            two_local_preds_l_probs,
            three_local_preds_l_probs,

            one_global_preds_l,
            two_global_preds_l,
            three_global_preds_l,

            reconstructed_single_concepts,
            reconstructed_single_concepts_preds,
            reconstructed_2_concepts,
            reconstructed_2_concepts_preds,
            reconstructed_3_concepts,
            reconstructed_3_concepts_preds,
            reconstructed_all_concepts,
            reconstructed_all_concepts_preds,

            label_maps,
            drift_ratios]

method_names = ["drift_localizer",
            "one_local_one_global_l",
            "one_local_l",
            "two_local_l",
            "three_local_l",
            "one_local_l_probs",
            "two_local_l_probs",
            "three_local_l_probs",

            "one_global_l",
            "two_global_l",
            "three_global_l",

            "one_local_one_global_preds_l",
            "one_local_preds_l",
            "two_local_preds_l",
            "three_local_preds_l",
            "one_local_preds_l_probs",
            "two_local_preds_l_probs",
            "three_local_preds_l_probs",

            "one_global_preds_l",
            "two_global_preds_l",
            "three_global_preds_l",
            "reconstructed_single_concepts",
            "reconstructed_single_concepts_preds",
            "reconstructed_2_concepts",
            "reconstructed_2_concepts_preds",
            "reconstructed_3_concepts",
            "reconstructed_3_concepts_preds",
            "reconstructed_all_concepts",
            "reconstructed_all_concepts_preds",
            "label_maps",
            "drift_ratios"]

with open('/content/drive/MyDrive/results/text_experiment_dbpedia.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['Method'] + [f'Run_{i+1}' for i in range(run_num)])
    for method, accuracies in zip(method_names, methods):
        writer.writerow([method] + accuracies)

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/drive/MyDrive/results/text_experiment_dbpedia.csv')
df = df.iloc[:29]

stats = {}
for method in df['Method']:
    accuracies = df[df['Method'] == method].drop('Method', axis=1).values.flatten().astype(float)
    mean = np.mean(accuracies)
    std = np.std(accuracies)
    stats[method] = (mean, std)

print(stats)

{'drift_localizer': (np.float64(0.8152000000000001), np.float64(0.08604949480128024)), 'one_local_one_global_l': (np.float64(0.8298666666666666), np.float64(0.07466952375486119)), 'one_local_l': (np.float64(0.8271999999999999), np.float64(0.07799532749537699)), 'two_local_l': (np.float64(0.7949333333333334), np.float64(0.07868429181086767)), 'three_local_l': (np.float64(0.7560000000000001), np.float64(0.09467605587240925)), 'one_local_l_probs': (np.float64(0.8240000000000001), np.float64(0.07719527907269401)), 'two_local_l_probs': (np.float64(0.7941333333333332), np.float64(0.06952077227547014)), 'three_local_l_probs': (np.float64(0.7509333333333333), np.float64(0.08657184556450466)), 'one_global_l': (np.float64(0.6530666666666667), np.float64(0.12548189776484361)), 'two_global_l': (np.float64(0.7370666666666666), np.float64(0.09362013078867648)), 'three_global_l': (np.float64(0.7845333333333333), np.float64(0.08967536512951096)), 'one_local_one_global_preds_l': (np.float64(0.905866666